# miRNA inference example 
## RCC scRNA-seq datasets - renal cell cancer (RCC). Accession GSE207493

Final model: stacking (Ridge): **TabPack (Muon) + DCNv2 + TabM**.

Eligible targets: **145** miRNAs (Optimal_K on TEST tune half: median bootstrap R² ≥ 0.5 on bulk and max SC cohort).

> Run Jupyter via `mirna-repo/run_jupyter.sh` (TabPack host venv, Python ≥3.12). The old `inference-gpu` Docker image (Python 3.10) cannot import TabPack.


In [2]:
import sys
import json
from pathlib import Path

# TabPack needs Python >=3.11 (star-subscript syntax). If you see 3.10 here,
# you are still on the OLD Docker Jupyter — open the host server from ./run_jupyter.sh.
print("Python:", sys.version)
assert sys.version_info >= (3, 11), (
    f"Need Python >=3.11 for TabPack, got {sys.version}. "
    "Stop Docker Jupyter on :8888 and run: ~/mirna-repo/run_jupyter.sh"
)

import pandas as pd

from preprocessor import SingleCell
from constants import (
    CONFIG_PATH,
    INFERENCE_DIR,
    INFERENCE_INPUT_DIR,
    INFERENCE_OUTPUT_DIR,
    MODELS_ROOT,
    STACK_MODELS,
    parse_prediction_config,
)

# processed inference input data: https://www.kaggle.com/datasets/ismailovaly/mirna-prediction-project
RCC_DIR = INFERENCE_INPUT_DIR
OUTPUT_DIR = INFERENCE_OUTPUT_DIR
FIG_PATH = INFERENCE_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RCC files:", sorted(p.name for p in RCC_DIR.glob("RCC_S*.csv")))
print("Config path:", CONFIG_PATH)
print("Models root:", MODELS_ROOT)
print("Stack models:", STACK_MODELS)
print("SingleCell module:", SingleCell.__module__)


Python: 3.12.9 (main, Mar 17 2025, 21:01:58) [Clang 20.1.0 ]
RCC files: ['RCC_S1.csv', 'RCC_S2.csv', 'RCC_S3.csv', 'RCC_S4.csv', 'RCC_S5.csv']
Config path: /home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/inference/prediction_config.json
Models root: /home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/train/results
Stack models: ('tabpack', 'dcnv2', 'tabm')
SingleCell module: preprocessor


# 1-7: prediction step by step: data preparation, normalization, KNN imputing / KNN pseudobulk sampling, prediction
# 8: prediction using ine method, all above steps wrapped 

## 1. Config

- **K1** — single-cell (after KNN impute)
- **K2…K10** — pseudobulk KNN (without imputing)

Config schema (v2): `eligible_mirs`, `cohorts` = list of names, per-cohort maps `K1`…`K10` with `features` / metrics.
Use `parse_prediction_config()` (or `sc.target_info`) — there is no top-level `targets` key.


In [3]:
cfg = json.loads(CONFIG_PATH.read_text())
eligible, cohorts, target_info = parse_prediction_config(cfg)

print("Eligible miRNAs:", len(eligible))
print("Cohort sizes:", {k: len(v) for k, v in cohorts.items()})
print("Assignment rule:", str(cfg.get("assignment_rule", ""))[:140], "...")

example = "hsa-mir-100-5p"
info = target_info[example]
print(f"\nExample {example}:")
print("  assigned_cohort:", info["assigned_cohort"])
print("  n_features:", len(info["genes"]))
print("  m_bulk / m_assigned:", info.get("test_bulk"), "/", info.get("test_optimal_k"))


Eligible miRNAs: 159
Cohort sizes: {'K1': 9, 'K2': 18, 'K3': 15, 'K4': 14, 'K5': 15, 'K10': 88}
Assignment rule:  ...

Example hsa-mir-100-5p:
  assigned_cohort: K1
  n_features: 433
  m_bulk / m_assigned: 0.79040789512393 / 0.9294086326835129


## 2. Load RCC dataset

Input: **cells × genes** CSV with columns `barcode`, `CellType` except for mRNA expression data (raw - counts)

In [4]:
SAMPLE = "RCC_S1"  # RCC_S1 … RCC_S5
INPUT_PATH = RCC_DIR / f"{SAMPLE}.csv"

# Для быстрой отладки: nrows=200. Для полного инференса: nrows=None
NROWS = 200

raw = pd.read_csv(INPUT_PATH, nrows=NROWS)
raw = raw.set_index("barcode")

print(raw.shape)
print(raw[["CellType"]].head())
print("ENSG columns:", sum(str(c).startswith("ENSG") for c in raw.columns))

(200, 33435)
                           CellType
barcode                            
AAACCCAGTAAGCAAT-1          T cells
AAACCCAGTCTGTGCG-1  Malignant cells
AAACCCAGTTAGAGAT-1          T cells
AAACCCATCACCTTGC-1          T cells
AAACGAAAGCGATTCT-1          T cells
ENSG columns: 33434


In [5]:
raw.head()

,CellType,ENSG00000243485,ENSG00000237613,ENSG00000186092,ENSG00000238009,ENSG00000239945,ENSG00000239906,ENSG00000241599,ENSG00000236601,ENSG00000284733,...,ENSG00000277196,ENSG00000277630,ENSG00000278384,ENSG00000278633,ENSG00000276345,ENSG00000277856,ENSG00000275063,ENSG00000271254,ENSG00000277475,ENSG00000268674
barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCAGTAAGCAAT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCAGTCTGTGCG-1,Malignant cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCAGTTAGAGAT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
AAACCCATCACCTTGC-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACGAAAGCGATTCT-1,T cells,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. `SingleCell` and `StackPredictor` classes

`SingleCell` wraps preprocessing (TPM, KNN impute, KNN pseudobulk) and `StackPredictor`
(TabPack + DCNv2 + TabM → Ridge).


In [6]:
sc = SingleCell(
    device="cuda",  # or "cpu"
    preload_models=False,
)

print("Eligible miRNAs:", len(sc.available_mirnas))
print("K1 miRNAs:", len(sc.mirnas_for_cohort("K1")), sc.mirnas_for_cohort("K1"))
print("K2 miRNAs:", len(sc.mirnas_for_pseudobulk_k(2)))
print("Cohorts:", {k: len(v) for k, v in sc.cohorts.items()})


Eligible miRNAs: 159
K1 miRNAs: 9 ['hsa-let-7b-5p', 'hsa-mir-100-5p', 'hsa-mir-134-3p', 'hsa-mir-142-3p', 'hsa-mir-20a-5p', 'hsa-mir-21-5p', 'hsa-mir-30a-3p', 'hsa-mir-335-5p', 'hsa-mir-449a']
K2 miRNAs: 18
Cohorts: {'K1': 9, 'K2': 18, 'K3': 15, 'K4': 14, 'K5': 15, 'K10': 88}


## 4. Preprocessing step by step (single-cell K1)

### Step 4.1 — align genes

`prepare_input()` adjusts matrix to fix set of mRNA **17 392 ENSG**, non existing genes filled by 0.

In [7]:
counts_gc = sc.prepare_input(raw)
print("genes × cells:", counts_gc.shape)
print("gene order fixed:", counts_gc.index[:3].tolist())

genes × cells: (17392, 200)
gene order fixed: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419']


### Step 4.2 — TPM and log2 normalization

Normalization: counts → RPK → TPM → `log2(TPM + 1)`.

In [8]:
log_tpm_gc = sc.TPM(counts_gc, enforce_mrna_standard=False)
log_tpm_cells = log_tpm_gc.T  # cells × genes — формат для stack

print("log2(TPM+1) cells × genes:", log_tpm_cells.shape)
log_tpm_cells.iloc[:3, :3]

✔ Found length for 17392/17392 genes (100.00%)
log2(TPM+1) cells × genes: (200, 17392)


,ENSG00000000003,ENSG00000000005,ENSG00000000419
barcode,,,
AAACCCAGTAAGCAAT-1,0.0,0.0,0.0
AAACCCAGTCTGTGCG-1,0.0,0.0,0.0
AAACCCAGTTAGAGAT-1,0.0,0.0,0.0


### Step 4.3 — KNN imputation (only for K1 single-cell)

Zeros in `log2(TPM+1)` replaces with average of k=5 nearest neighbors 

In [9]:
log_tpm_imputed_gc = sc.knn_impute_log_tpm(log_tpm_gc, knn_k=5)
log_tpm_imputed = log_tpm_imputed_gc.T

zeros_before = (log_tpm_cells == 0).sum().sum()
zeros_after = (log_tpm_imputed == 0).sum().sum()
print(f"zeros before/after impute: {zeros_before} → {zeros_after}")

Loading KNN reference...
KNN reference ready!
zeros before/after impute: 2966199 → 1263458


### Step 4.4 — Stack prediction (K1 cohort)

For each miRNA: **TabPack + DCNv2 + TabM → Ridge stack**


In [10]:
# A: step by step (as above) + predict
pred_manual = sc.predict(log_tpm_imputed, mirnas=sc.mirnas_for_cohort("K1"))
print("manual K1 predictions:", pred_manual.shape)

# B: one-shot from raw counts (recommended for K1)
pred_k1 = sc.predict_single_cell_knn_imputed(raw)
print("one-shot K1 predictions:", pred_k1.shape)
pred_k1.iloc[:5, :5]


Stack prediction for 9 miRNAs...
Loading final_train stack models (tabpack + dcnv2 + tabm)...
✔ Stack ready: 159 eligible miRNAs
manual K1 predictions: (200, 9)
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 9 miRNAs...
one-shot K1 predictions: (200, 9)


,hsa-let-7b-5p,hsa-mir-100-5p,hsa-mir-134-3p,hsa-mir-142-3p,hsa-mir-20a-5p
barcode,,,,,
AAACCCAGTAAGCAAT-1,13.308116,8.465096,0.025688,4.564407,13.325662
AAACCCAGTCTGTGCG-1,7.708419,7.945310,0.139983,3.832267,14.149895
AAACCCAGTTAGAGAT-1,10.824607,1.265426,0.057760,6.243568,12.746371
AAACCCATCACCTTGC-1,12.795567,10.321359,0.036582,5.478000,12.627148
AAACGAAAGCGATTCT-1,12.193323,9.989660,0.016323,4.468669,11.449169


## 5. Pseudobulk inference (K = 2, 3, 4, 5, 10)

For pseudobulk:
1. In PCA-space (log1p CPM, top HVG) find K nearest neighbors for each cell
2. Sum K cell raw counts → pseudobulk counts
3. TPM → stack **without** KNN impute

In [11]:
K_PB = 2 # example 
pred_pb_k2 = sc.predict_knn_pseudobulk(raw, K=K_PB)
print(f"PB K={K_PB}:", pred_pb_k2.shape)
print("miRNAs:", list(pred_pb_k2.columns[:5]), "...")
pred_pb_k2.iloc[:5, :5]

✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
PB K=2: (200, 18)
miRNAs: ['hsa-let-7d-5p', 'hsa-let-7i-5p', 'hsa-mir-1301-3p', 'hsa-mir-130a-3p', 'hsa-mir-135b-5p'] ...


,hsa-let-7d-5p,hsa-let-7i-5p,hsa-mir-1301-3p,hsa-mir-130a-3p,hsa-mir-135b-5p
barcode,,,,,
AAACCCAGTAAGCAAT-1,10.004285,8.111323,6.401043,3.461318,1.431046
AAACCCAGTCTGTGCG-1,7.444339,8.423554,6.246747,1.785949,6.389971
AAACCCAGTTAGAGAT-1,10.536684,10.997214,5.386059,5.009338,0.286757
AAACCCATCACCTTGC-1,10.563401,10.854077,5.408475,0.446886,0.000000
AAACGAAAGCGATTCT-1,10.899364,9.633400,4.905890,5.212992,0.000000


## 8. All in one method — `predict_all`

One method for all **145 eligible** miRNAs: K1 → single-cell + impute, K2…K10 → pseudobulk


In [12]:
SAMPLES = ["RCC_S1", "RCC_S2", "RCC_S3", "RCC_S4", "RCC_S5"]

for sample in SAMPLES:
    INPUT_PATH = RCC_DIR / f"{sample}.csv"
    raw_full = pd.read_csv(INPUT_PATH)
    raw_full = raw_full.set_index("barcode")

    pred_all = sc.predict_all(raw_full)
    out_path = OUTPUT_DIR / f"{sample}.csv"
    pred_all.to_csv(out_path)
    print(sample, pred_all.shape, "→", out_path)


Full inference: 6761 cells, 159 eligible miRNAs
  K1 single-cell + KNN impute: 9 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 9 miRNAs...
  K2 KNN pseudobulk (K=2): 18 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
  K3 KNN pseudobulk (K=3): 15 miRNAs
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 15 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 15 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 15 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 15 miRNAs...
✔ Found length for 17392/1739

# RESULTS

In [ ]:
from IPython.display import Image, display

display(Image(filename=FIG_PATH / "cancer_mirs.jpg"))

In [ ]:
display(Image(filename=FIG_PATH / "down.jpg"))

In [ ]:
display(Image(filename=FIG_PATH / "immune.jpg"))

In [ ]:
display(Image(filename=FIG_PATH / "vascular.jpg"))